In [1]:
# MCP SCENARIO: “Smart IT Helpdesk Assistant”
# 🧩 Scenario Background

# You are working in a company called ABC Corp.

# Employees face issues like:

# VPN not working
# Printer not responding
# Software errors

# 👉 Instead of calling IT support, employees use an AI Helpdesk Bot.

# 🤖 What this Bot Should Do

# When a user types a problem:

# Understand the issue
# Decide if a ticket is needed
# Identify:
# Category (Network / Hardware / General)
# Priority (High / Medium)
# Create a ticket
# Show confirmation
# 🧠 How MCP Fits Here
# Component	Role in Scenario
# Agent	Helpdesk Bot
# MCP Layer	Decision + Tool calling
# Tool	Ticket Creation System
# User	Employee

In [2]:
tickets_db = []

In [3]:
def create_ticket(issue, priority, category):
    """
    This function simulates a TOOL in MCP
    In real world → API / Database / ServiceNow
    """

    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket

In [4]:
def analyze_input(user_input):
    """
    Simulates how an LLM understands user input
    Extracts:
    - category
    - priority
    """

    text = user_input.lower()

    # 🔹 Category Detection
    if "vpn" in text:
        category = "network"
    elif "printer" in text:
        category = "hardware"
    elif "email" in text:
        category = "software"
    else:
        category = "general"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "slow" in text:
        priority = "low"
    else:
        priority = "medium"

    return category, priority


In [5]:
def should_call_tool(user_input):
    """
    Decides whether to call a tool or not
    This is MCP decision layer
    """

    keywords = ["issue", "problem", "ticket", "not working"]

    return any(word in user_input.lower() for word in keywords)


In [6]:
def mcp_agent(user_input):
    """
    This is the MAIN MCP FLOW
    It connects:
    Agent → Decision → Tool → Response
    """

    print("\n🧠 Agent received input:", user_input)

    # STEP 4.1: Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Tool call required")

        # STEP 4.2: Analyze input
        category, priority = analyze_input(user_input)

        print(f"📊 Extracted → Category: {category}, Priority: {priority}")

        # STEP 4.3: Prepare payload (MCP format)
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        # STEP 4.4: Call tool
        result = create_ticket(**payload)

        print("⚙️ Tool executed successfully")

        # STEP 4.5: Final response
        return f"""
        ✅ Ticket Created Successfully!

        Ticket ID: {result['ticket_id']}
        Issue: {result['issue']}
        Category: {result['category']}
        Priority: {result['priority']}
        """

    else:
        print("➡️ Decision: No tool needed (AI response)")

        return "🤖 AI Response: Please describe your issue clearly."

In [7]:
print("🚀 MCP Demo Started (Type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting MCP demo...")
        break

    response = mcp_agent(user_input)
    print(response)

🚀 MCP Demo Started (Type 'exit' to stop)



Enter your query:  VPN not Working?



🧠 Agent received input: VPN not Working?
➡️ Decision: Tool call required
📊 Extracted → Category: network, Priority: medium
📦 MCP Payload: {'issue': 'VPN not Working?', 'priority': 'medium', 'category': 'network'}
⚙️ Tool executed successfully

        ✅ Ticket Created Successfully!

        Ticket ID: INC1000
        Issue: VPN not Working?
        Category: network
        Priority: medium
        


Enter your query:  Software Errors



🧠 Agent received input: Software Errors
➡️ Decision: No tool needed (AI response)
🤖 AI Response: Please describe your issue clearly.


Enter your query:  exit


👋 Exiting MCP demo...


In [ ]:
# WITH API KEY

In [8]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# Load API key securely
api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=api_key)

In [9]:
tickets_db = []


def create_ticket(issue, priority, category):
    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket



def analyze_with_llm(user_input):
    """
    LLM decides:
    - should_create_ticket
    - category
    - priority
    """

    prompt = f"""
You are an IT helpdesk assistant.

Analyze the user issue and respond in JSON format:

{{
  "create_ticket": true/false,
  "category": "network/hardware/software/general",
  "priority": "high/medium/low"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # fast + powerful
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium"
        }

    return parsed


In [10]:
def mcp_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
✅ Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
"""

    else:
        return "🤖 AI Response: No ticket required. Try basic troubleshooting."

In [11]:
print("🚀 LLM MCP Helpdesk Started (type 'exit')\n")

while True:

    user_input = input("Enter issue: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

🚀 LLM MCP Helpdesk Started (type 'exit')



Enter issue:  Software Issues



🧠 Agent received: Software Issues
🤖 LLM Decision: {'create_ticket': True, 'category': 'software', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'Software Issues', 'priority': 'medium', 'category': 'software'}

✅ Ticket Created Successfully!

Ticket ID: INC1000
Issue: Software Issues
Category: software
Priority: medium



Enter issue:  VPN NOT WORKING



🧠 Agent received: VPN NOT WORKING
🤖 LLM Decision: {'create_ticket': True, 'category': 'network', 'priority': 'high'}
📦 MCP Payload: {'issue': 'VPN NOT WORKING', 'priority': 'high', 'category': 'network'}

✅ Ticket Created Successfully!

Ticket ID: INC1001
Issue: VPN NOT WORKING
Category: network
Priority: high



...


🧠 Agent received: MOUSE NOT WORKING
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'MOUSE NOT WORKING', 'priority': 'medium', 'category': 'general'}

✅ Ticket Created Successfully!

Ticket ID: INC1002
Issue: MOUSE NOT WORKING
Category: general
Priority: medium



Enter issue:  POWER CHARGING IS CAUSING PROBLEM



🧠 Agent received: POWER CHARGING IS CAUSING PROBLEM
🤖 LLM Decision: {'create_ticket': True, 'category': 'hardware', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'POWER CHARGING IS CAUSING PROBLEM', 'priority': 'medium', 'category': 'hardware'}

✅ Ticket Created Successfully!

Ticket ID: INC1003
Issue: POWER CHARGING IS CAUSING PROBLEM
Category: hardware
Priority: medium



Enter issue:  EXIT


👋 Exiting...


In [12]:
# MCP SCENARIO: “Smart HR Onboarding Assistant”
# 🧩 Scenario Background
# You are working in a company called XYZ Corp.
# New employees often face challenges during onboarding, such as:
# - Trouble accessing payroll portal
# - Confusion about leave policies
# - Difficulty setting up email accounts
# - Questions about training schedules
# 👉 Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

# 🤖 What this Bot Should Do
# When a new hire types a question/problem:
# - Understand the query (e.g., “I can’t log into payroll”)
# - Decide if escalation to HR is needed
# - Identify:
# - Category (Payroll / Policy / IT Setup / Training)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, step-by-step instructions)
# - Show confirmation and next steps

# 🧠 How MCP Fits Here
# |  |  | 
# |  |  | 
# |  |  | 
# |  |  | 
# |  |  | 



# This way, the MCP framework is reused in a Human Resources context, where the AI assistant streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.
# Would you like me to design another variation in a customer service setting (like retail or banking), so you can compare how MCP adapts across industries?

In [13]:
tickets_db = []  # Stores all HR tickets


# ============================================
# STEP 1: TOOL (HR SYSTEM / SERVICE)
# ============================================

def create_hr_ticket(issue, priority, category):
    """
    Simulates HR Ticket System (Tool)
    """

    ticket_id = f"HR{2000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# ============================================
# STEP 2: AGENT REASONING (LLM SIMULATION)
# ============================================

def analyze_input(user_input):
    """
    Extract category + priority from HR queries
    """

    text = user_input.lower()

    # 🔹 Category Detection
    if "payroll" in text or "salary" in text:
        category = "Payroll"
    elif "leave" in text or "policy" in text:
        category = "Policy"
    elif "email" in text or "login" in text or "account" in text:
        category = "IT Setup"
    elif "training" in text:
        category = "Training"
    else:
        category = "General"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text or "can't" in text:
        priority = "High"
    elif "confused" in text or "not sure" in text:
        priority = "Medium"
    else:
        priority = "Low"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE (MCP CORE)
# ============================================

def should_call_tool(user_input):
    """
    Decide whether to escalate to HR or not
    """

    keywords = ["can't", "not able", "issue", "problem", "error"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: KNOWLEDGE BASE (AI RESPONSE)
# ============================================

def get_guidance(category):
    """
    Provides instant help without ticket
    """

    if category == "Payroll":
        return "💰 Payroll Info: Salary is processed on last working day. Check payroll portal."

    elif category == "Policy":
        return "📜 Leave Policy: You get 20 paid leaves annually. Apply via HR portal."

    elif category == "IT Setup":
        return "💻 IT Setup: Use your employee ID to login. Reset password if needed."

    elif category == "Training":
        return "📚 Training: Check LMS portal for onboarding schedule."

    else:
        return "ℹ️ Please contact HR for more details."


# ============================================
# STEP 5: MCP ORCHESTRATOR
# ============================================

def mcp_hr_agent(user_input):
    """
    Main MCP Flow for HR Bot
    """

    print("\n🧠 HR Agent received:", user_input)

    # STEP 5.1: Analyze input
    category, priority = analyze_input(user_input)

    print(f"📊 Category: {category}, Priority: {priority}")

    # STEP 5.2: Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Escalate to HR (Create Ticket)")

        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        result = create_hr_ticket(**payload)

        return f"""
        ✅ HR Ticket Created!

        Ticket ID: {result['ticket_id']}
        Issue: {result['issue']}
        Category: {result['category']}
        Priority: {result['priority']}

        📩 HR will contact you shortly.
        """

    else:
        print("➡️ Decision: Provide Instant Help")

        guidance = get_guidance(category)

        return f"""
        🤖 Instant Help:

        {guidance}

        👉 If issue persists, type 'issue' to raise a ticket.
        """


# ============================================
# STEP 6: RUN LOOP
# ============================================

print("🚀 Smart HR Onboarding Assistant Started (type 'exit')\n")

while True:

    user_input = input("Ask HR Bot: ")

    if user_input.lower() == "exit":
        print("👋 Exiting HR Assistant...")
        break

    response = mcp_hr_agent(user_input)
    print(response)

🚀 Smart HR Onboarding Assistant Started (type 'exit')



Ask HR Bot:  I can't login to payroll



🧠 HR Agent received: I can't login to payroll
📊 Category: Payroll, Priority: High
➡️ Decision: Escalate to HR (Create Ticket)

        ✅ HR Ticket Created!

        Ticket ID: HR2000
        Issue: I can't login to payroll
        Category: Payroll
        Priority: High

        📩 HR will contact you shortly.
        


Ask HR Bot:  exit


👋 Exiting HR Assistant...


In [ ]:
# WITH API

In [15]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# Load API key securely
api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=api_key)


In [16]:


tickets_db = []


# ============================================
# STEP 1: TOOL (HR SYSTEM)
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"HR{2000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: LLM ANALYSIS (HR PROMPT)
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are an HR onboarding assistant.

Analyze the employee query and return ONLY valid JSON:

{{
  "create_ticket": true/false,
  "category": "Payroll/Policy/IT Setup/Training/General",
  "priority": "High/Medium/Low"
}}

Rules:
- Create ticket if it's a login issue, access problem, or blocking issue
- No ticket for general questions (policies, info)
- Payroll issues → High priority
- Login issues → High priority
- General queries → Low priority

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content.strip()

    print("📥 Raw LLM Output:", output)

    # 🔥 SAFE JSON PARSING (VERY IMPORTANT)
    import json
    try:
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "General",
            "priority": "Medium"
        }

    return parsed


# ============================================
# STEP 3: KNOWLEDGE BASE
# ============================================

def get_guidance(category):

    if category == "Payroll":
        return "💰 Salary is processed at the end of the month. Check payroll portal."

    elif category == "Policy":
        return "📜 You get 20 paid leaves annually. Apply via HR portal."

    elif category == "IT Setup":
        return "💻 Use employee ID to login. Try resetting password."

    elif category == "Training":
        return "📚 Check LMS portal for onboarding training schedule."

    else:
        return "ℹ️ Please contact HR for more details."


# ============================================
# STEP 4: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 HR Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
✅ HR Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

📩 HR team will contact you soon.
"""

    else:
        guidance = get_guidance(decision["category"])

        return f"""
🤖 Instant Help:

{guidance}

👉 If issue persists, raise a ticket.
"""


# ============================================
# STEP 5: RUN LOOP
# ============================================

print("🚀 LLM HR Onboarding Assistant Started (type 'exit')\n")

while True:

    user_input = input("Ask HR Bot: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

🚀 LLM HR Onboarding Assistant Started (type 'exit')



Ask HR Bot:  what is a leave policy?



🧠 HR Agent received: what is a leave policy?
📥 Raw LLM Output: ```json
{
  "create_ticket": false,
  "category": "Policy",
  "priority": "Low"
}
```
🤖 LLM Decision: {'create_ticket': True, 'category': 'General', 'priority': 'Medium'}
📦 MCP Payload: {'issue': 'what is a leave policy?', 'priority': 'Medium', 'category': 'General'}

✅ HR Ticket Created!

Ticket ID: HR2000
Category: General
Priority: Medium

📩 HR team will contact you soon.



Ask HR Bot:  Input: I can't login to payroll



🧠 HR Agent received: Input: I can't login to payroll
📥 Raw LLM Output: ```json
{
  "create_ticket": true,
  "category": "IT Setup",
  "priority": "High"
}
```
🤖 LLM Decision: {'create_ticket': True, 'category': 'General', 'priority': 'Medium'}
📦 MCP Payload: {'issue': "Input: I can't login to payroll", 'priority': 'Medium', 'category': 'General'}

✅ HR Ticket Created!

Ticket ID: HR2001
Category: General
Priority: Medium

📩 HR team will contact you soon.



Ask HR Bot:  exit


👋 Exiting...


In [17]:
# MCP SCENARIO: “Smart Banking Support Assistant”
# 🧩 Scenario Background
# You are working in a company called FinTrust Bank.
# Customers often face issues such as:
# - Credit card not working
# - Trouble with online banking login
# - Queries about loan status
# - Transaction disputes
# 👉 Instead of calling customer care, customers use an AI Banking Support Bot.

# 🤖 What this Bot Should Do
# When a customer types a problem:
# - Understand the issue (e.g., “My card was declined”)
# - Decide if escalation to a human agent is needed
# - Identify:
# - Category (Card Services / Online Banking / Loans / Transactions)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, troubleshooting steps, policy info)
# - Show confirmation and next steps

# 🧠 How MCP Fits Here
# |  |  | 
# |  |  | 
# |  |  | 
# |  |  | 
# |  |  | 



# This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
# Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?

In [18]:
tickets_db = []


# ============================================
# TOOL (BANK SYSTEM)
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"BNK{3000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# ANALYSIS (RULE-BASED / SIMPLIFIED LLM)
# ============================================

def analyze_input(user_input):

    text = user_input.lower()

    # Category
    if "card" in text:
        category = "Card Services"
    elif "login" in text or "banking" in text:
        category = "Online Banking"
    elif "loan" in text:
        category = "Loans"
    elif "transaction" in text or "fraud" in text:
        category = "Transactions"
    else:
        category = "General"

    # Priority
    if "fraud" in text or "declined" in text:
        priority = "High"
    else:
        priority = "Medium"

    return category, priority


# ============================================
# DECISION
# ============================================

def should_create_ticket(user_input):

    keywords = ["not working", "declined", "fraud", "issue", "error"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# KNOWLEDGE BASE
# ============================================

def get_guidance(category):

    if category == "Card Services":
        return "💳 Ensure card is active and has sufficient balance."

    elif category == "Online Banking":
        return "🔐 Try resetting password using 'Forgot Password'."

    elif category == "Loans":
        return "🏦 Check loan status in 'My Loans' section."

    elif category == "Transactions":
        return "💰 Check transaction history in your statement."

    else:
        return "ℹ️ Please contact support."


# ============================================
# MCP AGENT
# ============================================

def mcp_banking_agent(user_input):

    print("\n🧠 Banking Agent received:", user_input)

    category, priority = analyze_input(user_input)

    if should_create_ticket(user_input):

        result = create_ticket(user_input, priority, category)

        return f"""
✅ Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {category}
Priority: {priority}

📞 Our banking team will contact you soon.
"""

    else:

        guidance = get_guidance(category)

        return f"""
🤖 Instant Help:

{guidance}
"""


# ============================================
# RUN LOOP
# ============================================

print("🏦 Banking Support Bot Started (type 'exit')\n")

while True:

    user_input = input("Enter your issue: ")

    if user_input.lower() == "exit":
        break

    print(mcp_banking_agent(user_input))

🏦 Banking Support Bot Started (type 'exit')



Enter your issue:  Credit card Not working



🧠 Banking Agent received: Credit card Not working

✅ Ticket Created!

Ticket ID: BNK3000
Category: Card Services
Priority: Medium

📞 Our banking team will contact you soon.



Enter your issue:  Facing issue in money transaction



🧠 Banking Agent received: Facing issue in money transaction

✅ Ticket Created!

Ticket ID: BNK3001
Category: Transactions
Priority: Medium

📞 Our banking team will contact you soon.



Enter your issue:  Fraud person is trying to access my account



🧠 Banking Agent received: Fraud person is trying to access my account

✅ Ticket Created!

Ticket ID: BNK3002
Category: Transactions
Priority: High

📞 Our banking team will contact you soon.



Enter your issue:  exit


In [20]:
# With Api

In [21]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# Load API key securely
api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=api_key)

tickets_db = []


# ============================================
# STEP 1: TOOL (BANK SYSTEM)
# ============================================

def create_ticket(issue, priority, category):

    ticket_id = f"BNK{3000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: LLM ANALYSIS (GROQ)
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are a banking support assistant.

Analyze the user query and return ONLY valid JSON:

{{
  "create_ticket": true/false,
  "category": "Card Services/Online Banking/Loans/Transactions/General",
  "priority": "High/Medium"
}}

Rules:
- Card declined / fraud → High priority + ticket
- Login issues → High priority + ticket
- Transaction dispute → High priority + ticket
- Loan queries → No ticket (FAQ)
- General queries → No ticket

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content.strip()
    print("📥 LLM Raw Output:", output)

    # SAFE PARSING
    import json, re

    match = re.search(r"\{.*\}", output, re.DOTALL)

    if match:
        try:
            parsed = json.loads(match.group())
        except:
            parsed = {"create_ticket": True, "category": "General", "priority": "Medium"}
    else:
        parsed = {"create_ticket": True, "category": "General", "priority": "Medium"}

    return parsed


# ============================================
# STEP 3: KNOWLEDGE BASE
# ============================================

def get_guidance(category):

    if category == "Card Services":
        return "💳 Check if your card is active and has sufficient balance."

    elif category == "Online Banking":
        return "🔐 Reset password using 'Forgot Password' option."

    elif category == "Loans":
        return "🏦 Check loan status in your banking dashboard."

    elif category == "Transactions":
        return "💰 Review transaction history and verify details."

    else:
        return "ℹ️ Please contact bank support."


# ============================================
# STEP 4: MCP AGENT
# ============================================

def mcp_banking_agent(user_input):

    print("\n🧠 Banking Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
✅ Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

📞 Bank support will contact you shortly.
"""

    else:

        guidance = get_guidance(decision["category"])

        return f"""
🤖 Instant Help:

{guidance}

👉 If issue persists, raise a ticket.
"""


# ============================================
# STEP 5: RUN LOOP
# ============================================

print("🏦 Groq MCP Banking Bot Started (type 'exit')\n")

while True:

    user_input = input("Enter your issue: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_banking_agent(user_input)
    print(response)

🏦 Groq MCP Banking Bot Started (type 'exit')



Enter your issue:  My credit card is declined



🧠 Banking Agent received: My credit card is declined
📥 LLM Raw Output: ```json
{
  "create_ticket": true,
  "category": "Card Services",
  "priority": "High"
}
```
🤖 LLM Decision: {'create_ticket': True, 'category': 'Card Services', 'priority': 'High'}
📦 MCP Payload: {'issue': 'My credit card is declined', 'priority': 'High', 'category': 'Card Services'}

✅ Ticket Created Successfully!

Ticket ID: BNK3000
Category: Card Services
Priority: High

📞 Bank support will contact you shortly.



Enter your issue:  whats my loan status?



🧠 Banking Agent received: whats my loan status?
📥 LLM Raw Output: ```json
{
  "create_ticket": false,
  "category": "Loans",
  "priority": "Medium"
}
```
🤖 LLM Decision: {'create_ticket': False, 'category': 'Loans', 'priority': 'Medium'}

🤖 Instant Help:

🏦 Check loan status in your banking dashboard.

👉 If issue persists, raise a ticket.



Enter your issue:  exit


👋 Exiting...


In [22]:
# user ----- llm ------ mcp(client) ----- mcp(server) ------ github Api

In [ ]:
# Create a  Weather Tool MCP Server that any AI agent can use with sample use case

In [48]:
import json
import re
import gradio as gr
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

# Load API key
api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)

# ============================================
# TOOL 1: WEATHER
# ============================================

weather_data = {
    "delhi": "32°C, Sunny",
    "mumbai": "28°C, Humid",
    "london": "15°C, Cloudy"
}

def get_weather(city):
    return weather_data.get(city.lower(), "Weather data not available")

# ============================================
# TOOL 2: NEWS
# ============================================

news_data = {
    "india": "India: Tech sector growing rapidly 🚀",
    "world": "Global markets show mixed trends 📊",
    "sports": "India wins thrilling cricket match 🏏"
}

def get_news(topic):
    return news_data.get(topic.lower(), "No news available")

# ============================================
# JSON EXTRACTOR
# ============================================

def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            return None
    return None

# ============================================
# LLM DECISION (MCP BRAIN)
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are an AI assistant with tools.

Decide which tool to use.

Return ONLY JSON:

{{
  "tool": "weather/news/none",
  "input": "city or topic",
  "response": "normal reply if no tool"
}}

User Query: "{user_input}"
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        output = response.choices[0].message.content
        parsed = extract_json(output)

    except Exception as e:
        print("Error:", e)
        parsed = None

    if not parsed:
        parsed = {
            "tool": "none",
            "input": "",
            "response": "Sorry, I couldn't understand."
        }

    return parsed

# ============================================
# MCP AGENT
# ============================================

def mcp_multi_tool_agent(user_input):

    decision = analyze_with_llm(user_input)

    tool = decision.get("tool")
    tool_input = decision.get("input")

    # 🔹 WEATHER TOOL
    if tool == "weather":
        if not tool_input:
            return "❌ Please specify a city."
        result = get_weather(tool_input)
        return f"🌦️ Weather in {tool_input.title()}: {result}"

    # 🔹 NEWS TOOL
    elif tool == "news":
        if not tool_input:
            return "❌ Please specify a topic (india/world/sports)."
        result = get_news(tool_input)
        return f"📰 News ({tool_input.title()}): {result}"

    # 🔹 NO TOOL
    else:
        return decision.get("response")


# ============================================
# GRADIO UI
# ============================================

def chatbot_response(message, history):
    return mcp_multi_tool_agent(message)

chat_ui = gr.ChatInterface(
    fn=chatbot_response,
    title="🤖 Multi-Tool MCP Assistant",
    description="Ask weather or news (e.g., 'Weather in Delhi', 'News about India')"
)

chat_ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
